In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score

In [3]:
df = pd.read_csv('bagging_ex.csv')
df

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0
...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110,264,0,Normal,132,N,1.2,Flat,1
914,68,M,ASY,144,193,1,Normal,141,N,3.4,Flat,1
915,57,M,ASY,130,131,0,Normal,115,Y,1.2,Flat,1
916,57,F,ATA,130,236,0,LVH,174,N,0.0,Flat,1


In [4]:
df.isna().sum()

Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
dtype: int64

In [42]:
df.describe()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,53.510893,132.396514,198.799564,0.233115,136.809368,0.887364,0.553377
std,9.432617,18.514154,109.384145,0.423046,25.460334,1.066570,0.497414
min,28.000000,0.000000,0.000000,0.000000,60.000000,-2.600000,0.000000
25%,47.000000,120.000000,173.250000,0.000000,120.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,0.000000,138.000000,0.600000,1.000000
75%,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000,1.000000
max,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000,1.000000


In [6]:
df['HeartDisease'].value_counts()

HeartDisease
1    508
0    410
Name: count, dtype: int64

In [7]:
from scipy import stats
df_numeric = df.select_dtypes(include=[np.number]) 
df_clean = df[(np.abs(stats.zscore(df_numeric))<3).all(axis = 1)]
df_clean

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0
...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110,264,0,Normal,132,N,1.2,Flat,1
914,68,M,ASY,144,193,1,Normal,141,N,3.4,Flat,1
915,57,M,ASY,130,131,0,Normal,115,Y,1.2,Flat,1
916,57,F,ATA,130,236,0,LVH,174,N,0.0,Flat,1


In [34]:
df_clean.describe()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
count,899.000000,899.000000,899.000000,899.000000,899.000000,899.000000,899.000000,899.000000,899.000000,899.000000,899.000000,899.000000
mean,53.497219,0.789766,0.785317,132.027809,198.005562,0.232481,0.991101,136.917686,0.403782,0.861513,1.373749,0.547275
std,9.456073,0.407701,0.956496,17.120895,107.157779,0.422649,0.629569,25.356740,0.490928,1.007626,0.601057,0.498037
min,28.000000,0.000000,0.000000,80.000000,0.000000,0.000000,0.000000,63.000000,0.000000,-2.000000,0.000000,0.000000
25%,47.000000,1.000000,0.000000,120.000000,174.500000,0.000000,1.000000,120.000000,0.000000,0.000000,1.000000,0.000000
50%,54.000000,1.000000,0.000000,130.000000,222.000000,0.000000,1.000000,138.000000,0.000000,0.500000,1.000000,1.000000
75%,60.000000,1.000000,2.000000,140.000000,266.000000,0.000000,1.000000,156.000000,1.000000,1.500000,2.000000,1.000000
max,77.000000,1.000000,3.000000,185.000000,518.000000,1.000000,2.000000,202.000000,1.000000,4.000000,2.000000,1.000000


In [9]:
col_to_numeric = ['Sex','ChestPainType','RestingECG','ExerciseAngina',"ST_Slope"]

In [10]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_clean[col_to_numeric] = df_clean[col_to_numeric].apply(le.fit_transform)
df_clean

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,1,1,140,289,0,1,172,0,0.0,2,0
1,49,0,2,160,180,0,1,156,0,1.0,1,1
2,37,1,1,130,283,0,2,98,0,0.0,2,0
3,48,0,0,138,214,0,1,108,1,1.5,1,1
4,54,1,2,150,195,0,1,122,0,0.0,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,1,3,110,264,0,1,132,0,1.2,1,1
914,68,1,0,144,193,1,1,141,0,3.4,1,1
915,57,1,0,130,131,0,1,115,1,1.2,1,1
916,57,0,1,130,236,0,0,174,0,0.0,1,1


In [11]:
x = df_clean.drop('HeartDisease',axis='columns')
y = df_clean['HeartDisease']

In [12]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)
x_scaled


array([[-1.42815446,  0.515943  ,  0.2245723 , ..., -0.8229452 ,
        -0.85546862,  1.04249607],
       [-0.47585532, -1.93819859,  1.27063705, ..., -0.8229452 ,
         0.13751561, -0.62216462],
       [-1.7455875 ,  0.515943  ,  0.2245723 , ..., -0.8229452 ,
        -0.85546862,  1.04249607],
       ...,
       [ 0.3706328 ,  0.515943  , -0.82149245, ...,  1.21514774,
         0.33611246, -0.62216462],
       [ 0.3706328 , -1.93819859,  0.2245723 , ..., -0.8229452 ,
        -0.85546862, -0.62216462],
       [-1.63977649,  0.515943  ,  1.27063705, ..., -0.8229452 ,
        -0.85546862,  1.04249607]], shape=(899, 11))

In [14]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x_scaled,y,stratify=y,random_state=30)

In [15]:
X_train.shape

(674, 11)

In [17]:
X_test.shape

(225, 11)

In [18]:
y_train.value_counts()

HeartDisease
1    369
0    305
Name: count, dtype: int64

## train using stand alone model

In [25]:
scores = cross_val_score(DecisionTreeClassifier(),x,y,cv=5)
scores

array([0.78333333, 0.79444444, 0.78888889, 0.76111111, 0.61452514])

In [29]:
scores.mean()

np.float64(0.7484605834885165)

In [50]:
scores1 = cross_val_score(SVC(),x,y,cv=5)
scores1

array([0.62222222, 0.78888889, 0.68888889, 0.72222222, 0.62011173])

In [51]:
scores1.mean()

np.float64(0.6884667908131595)

## Using Bagging

In [52]:
from sklearn.ensemble import BaggingClassifier

bagging = BaggingClassifier(
    n_estimators=100,
    max_samples=0.8,
    estimator=SVC(),
    oob_score=True,
    random_state=0
    )
bagging.fit(X_train,y_train)
bagging.oob_score_

0.8783382789317508

In [53]:
bagging2 = BaggingClassifier(
    n_estimators=100,
    estimator=DecisionTreeClassifier(),
    max_samples=0.8,
    oob_score=True,
    random_state=0
    )
bagging2.fit(X_train,y_train)
bagging2.oob_score_

0.8427299703264095

## Train a model using Random Forest which itself uses bagging underneath

In [54]:
forest = RandomForestClassifier()
scores3 = cross_val_score(RandomForestClassifier(),x,y,cv=5)
scores3

array([0.87222222, 0.85555556, 0.83333333, 0.83888889, 0.77094972])

In [55]:
scores3.mean()

np.float64(0.8341899441340782)